# MiniMax H3 refine (stage 2 of the three-stage route)

Run by `scripts/refine.py`; the settings come in the environment (`R_*`). It stretches a stage-1 draft
to the refine canvas, encodes it with the H3 video VAE, and re-denoises only the low-noise tail: `R_STEPS`
steps evenly spaced on H3's base grid from `R_SIGMA_START` down to 0, mapped through the video shift 12.
The first frame is the keyframe at frame 0, as in the draft. Audio is not kept (the pipeline drops it).
The graph uses core ComfyUI nodes only and follows the refiner of Henge (alisson-anjos/Henge, MIT,
`backend/workflows.py` `full_refiner_graph`). See `references/three-stage.md`.

In [ ]:
import os, shutil, subprocess, sys, json, math, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), "this notebook needs a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0), "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "CUDA:", torch.version.cuda)
COMFY = Path("/content/ComfyUI")
VHS = COMFY / "custom_nodes" / "ComfyUI-VideoHelperSuite"
MARKER = Path("/content/.h3_refine_setup_v1")
if not MARKER.exists():
    if not COMFY.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt"), "huggingface_hub", "hf_xet", "requests"], check=True)
    if not VHS.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git", str(VHS)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(VHS / "requirements.txt")], check=True)
    MARKER.touch()
print("ComfyUI:", COMFY)

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "Comfy-Org/MiniMax-H3"
# The same weights the draft notebook picks: INT8 ConvRot on CUDA 13, FP8 scaled otherwise.
DIFFUSION = ("minimax_h3_fl2va_pruned_int8_convrot.safetensors" if (torch.version.cuda or "").startswith("13.")
             else "minimax_h3_fl2va_pruned_fp8_scaled.safetensors")
TEXT_ENCODER = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
VIDEO_VAE = "minimax_h3_video_vae_fp16.safetensors"
for folder, name in (("diffusion_models", DIFFUSION), ("text_encoders", TEXT_ENCODER), ("vae", VIDEO_VAE)):
    target = COMFY / "models" / folder / name
    if not target.is_file() or target.stat().st_size == 0:
        hf_hub_download(repo_id=REPO, filename=f"{folder}/{name}", local_dir=COMFY / "models")
    print(folder, name, f"{target.stat().st_size / 2**30:.2f} GiB")

In [ ]:
import requests
BASE_URL = "http://127.0.0.1:8188"
LOG_PATH = Path("/content/comfyui_refine.log")
try:
    requests.get(BASE_URL + "/system_stats", timeout=2).raise_for_status()
    print("using the running ComfyUI")
except requests.RequestException:
    LOG_FILE = open(LOG_PATH, "a")
    COMFY_PROCESS = subprocess.Popen([sys.executable, str(COMFY / "main.py"), "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
                                     cwd=COMFY, stdout=LOG_FILE, stderr=subprocess.STDOUT)
    for _ in range(180):
        if COMFY_PROCESS.poll() is not None:
            raise RuntimeError("ComfyUI exited:\n" + LOG_PATH.read_text()[-4000:])
        try:
            requests.get(BASE_URL + "/system_stats", timeout=2).raise_for_status()
            break
        except requests.RequestException:
            time.sleep(2)
    else:
        raise TimeoutError("ComfyUI did not start")
OBJECT_INFO = requests.get(BASE_URL + "/object_info", timeout=60).json()
NEEDED = {"UNETLoader", "CLIPLoader", "VAELoader", "LoadImage", "VHS_LoadVideoPath", "ImageScale", "VAEEncodeTiled",
          "EmptyMiniMaxH3LatentAV", "LTXVSeparateAVLatent", "LTXVConcatAVLatent", "MiniMaxH3ImageToVideo",
          "BasicGuider", "RandomNoise", "KSamplerSelect", "ManualSigmas", "SamplerCustomAdvanced", "VAEDecodeTiled",
          "VHS_VideoCombine", "MiniMaxH3AddGuide"}
missing = NEEDED - OBJECT_INFO.keys()
assert not missing, f"ComfyUI lacks nodes: {sorted(missing)}"
print("nodes ok")

In [ ]:
WIDTH = int(os.environ.get("R_WIDTH", "1312"))
HEIGHT = int(os.environ.get("R_HEIGHT", "736"))
SIGMA_START = float(os.environ.get("R_SIGMA_START", "0.5"))
STEPS = int(os.environ.get("R_STEPS", "10"))
SEED = int(os.environ.get("R_SEED", "0"))
SAMPLER = os.environ.get("R_SAMPLER", "res_multistep")
# "draft": the whole stretched draft is also a MiniMaxH3AddGuide clip at frame 0, so every frame is
# anchored, not only the first (Henge anchors its refine to the source the same way).
GUIDE = os.environ.get("R_GUIDE", "none")
EXPLICIT_SIGMAS = os.environ.get("R_SIGMAS", "").strip()
INPUT_VIDEO = os.environ.get("R_INPUT_VIDEO")
FIRST_FRAME = os.environ.get("R_FIRST_FRAME")
OUTPUT_PATH = Path(os.environ.get("R_OUTPUT_PATH", "/content/h3_refined.mp4"))
PROMPT = Path(os.environ.get("R_PROMPT_FILE")).read_text(encoding="utf-8")
SHIFT = 12.0  # H3's video flow shift (ComfyUI supported_models: shift 12.0, audio_shift 3.0)
assert WIDTH % 32 == 0 and HEIGHT % 32 == 0 and WIDTH * HEIGHT <= 1344 * 768, "canvas: multiples of 32 within 1344x768"
assert 0 < SIGMA_START < 1 and STEPS >= 1, "R_SIGMA_START in (0, 1), R_STEPS >= 1"
assert GUIDE in {"none", "draft"}, "R_GUIDE is none or draft"
probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_frames", "-show_entries", "stream=nb_read_frames",
                        "-of", "csv=p=0", INPUT_VIDEO], capture_output=True, text=True, check=True)
LENGTH = int(probe.stdout.strip())
assert (LENGTH - 5) % 17 == 0, f"the draft has {LENGTH} frames; H3 clips have 17k+5"
print("REFINE_FRAMES:", LENGTH)

# STEPS evenly spaced on the base grid from SIGMA_START to 0, through the shift: the steps all land
# where detail is made. "The last 10 of 20 steps" would start near sigma 0.92 and redraw the motion.
base0 = SIGMA_START / (SHIFT + SIGMA_START * (1.0 - SHIFT))
SIGMAS = ", ".join(f"{SHIFT * t / (1.0 + (SHIFT - 1.0) * t):.6f}" for t in (base0 * (1 - i / STEPS) for i in range(STEPS + 1)))
if EXPLICIT_SIGMAS:
    # A fixed list (e.g. Henge's three-step 0.909375, 0.857143, 0.666667, 0) replaces the computed one.
    values = [float(v) for v in EXPLICIT_SIGMAS.split(",")]
    assert len(values) >= 2 and values[-1] == 0 and all(a > b for a, b in zip(values, values[1:])) and values[0] < 1, "R_SIGMAS must fall to 0"
    SIGMAS = ", ".join(f"{v:.6f}" for v in values)
print("REFINE_SIGMAS:", SIGMAS)

(COMFY / "input").mkdir(exist_ok=True)
shutil.copy2(FIRST_FRAME, COMFY / "input" / "h3_refine_first_frame.png")


def node(kind, **inputs):
    # Required widgets left out take the node's own default, so a VHS update that adds one still runs.
    for name, item in OBJECT_INFO[kind]["input"].get("required", {}).items():
        if name not in inputs and isinstance(item, list) and len(item) > 1 and isinstance(item[1], dict) and "default" in item[1]:
            inputs[name] = item[1]["default"]
    return {"class_type": kind, "inputs": inputs}


graph = {
    "1": node("UNETLoader", unet_name=DIFFUSION, weight_dtype="default"),
    "3": node("CLIPLoader", clip_name=TEXT_ENCODER, type="minimax", device="default"),
    "4": node("VAELoader", vae_name=VIDEO_VAE),
    "20": node("VHS_LoadVideoPath", video=INPUT_VIDEO, force_rate=0, custom_width=0, custom_height=0,
               frame_load_cap=0, skip_first_frames=0, select_every_nth=1),
    "21": node("ImageScale", image=["20", 0], upscale_method="lanczos", width=WIDTH, height=HEIGHT, crop="disabled"),
    "22": node("VAEEncodeTiled", pixels=["21", 0], vae=["4", 0], tile_size=512, overlap=64, temporal_size=64, temporal_overlap=8),
    "23": node("EmptyMiniMaxH3LatentAV", width=WIDTH, height=HEIGHT, length=LENGTH),
    "24": node("LTXVSeparateAVLatent", av_latent=["23", 0]),
    "25": node("LTXVConcatAVLatent", video_latent=["22", 0], audio_latent=["24", 1]),
    "6": node("LoadImage", image="h3_refine_first_frame.png"),
    "7": node("MiniMaxH3ImageToVideo", clip=["3", 0], vae=["4", 0], first_frame=["6", 0], prompt=PROMPT,
              width=WIDTH, height=HEIGHT, length=LENGTH),
    "8": node("BasicGuider", model=["1", 0], conditioning=["27", 0] if GUIDE == "draft" else ["7", 0]),
    "9": node("RandomNoise", noise_seed=SEED),
    "10": node("KSamplerSelect", sampler_name=SAMPLER),
    "11": node("ManualSigmas", sigmas=SIGMAS),
    "12": node("SamplerCustomAdvanced", noise=["9", 0], guider=["8", 0], sampler=["10", 0], sigmas=["11", 0], latent_image=["25", 0]),
    "26": node("LTXVSeparateAVLatent", av_latent=["12", 0]),
    "13": node("VAEDecodeTiled", samples=["26", 0], vae=["4", 0], tile_size=512, overlap=64, temporal_size=64, temporal_overlap=8),
    **({"27": node("MiniMaxH3AddGuide", positive=["7", 0], latent=["25", 0], vae=["4", 0], image=["21", 0], frame_idx=0)}
       if GUIDE == "draft" else {}),
    "15": node("VHS_VideoCombine", images=["13", 0], frame_rate=24, loop_count=0, filename_prefix="h3_refine",
               format="video/h264-mp4", pix_fmt="yuv420p", crf=16, save_metadata=False, trim_to_audio=False,
               pingpong=False, save_output=True),
}
started = time.monotonic()
result = requests.post(BASE_URL + "/prompt", json={"prompt": graph, "client_id": "h3-refine"}, timeout=60)
if not result.ok:
    raise RuntimeError(f"submit failed HTTP {result.status_code}: {result.text[:4000]}")
job = result.json()
if job.get("node_errors"):
    raise RuntimeError("node errors: " + json.dumps(job["node_errors"], ensure_ascii=False)[:4000])
prompt_id = job["prompt_id"]
deadline = time.monotonic() + float(os.environ.get("R_JOB_TIMEOUT_SECONDS", "3600"))
last_report = 0
while time.monotonic() < deadline:
    history = requests.get(BASE_URL + f"/history/{prompt_id}", timeout=30).json()
    if prompt_id in history:
        run = history[prompt_id]
        if run.get("status", {}).get("status_str") == "error":
            raise RuntimeError("refine failed: " + json.dumps(run.get("status", {}), ensure_ascii=False)[:4000] + "\n" + LOG_PATH.read_text()[-3000:])
        if "15" in run.get("outputs", {}):
            break
    if time.monotonic() - last_report >= 60:
        print("still refining…", round(time.monotonic() - started), "s")
        last_report = time.monotonic()
    time.sleep(5)
else:
    raise TimeoutError("refine timed out")
info = next(item for item in run["outputs"]["15"]["gifs"] if item["filename"].endswith(".mp4"))
video_path = COMFY / "output" / info.get("subfolder", "") / info["filename"]
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(video_path, OUTPUT_PATH)
print("REFINE_SEED_USED:", SEED, "GUIDE:", GUIDE, "SAMPLER:", SAMPLER)
print("REFINE_SECONDS:", round(time.monotonic() - started, 1))
print("REFINE_OUTPUT:", OUTPUT_PATH, OUTPUT_PATH.stat().st_size)